In [1]:
!pip install -q google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 25.3.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.10.0 wh

In [2]:
import os
import time
import re
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

In [3]:
try:
    user_secrets = UserSecretsClient()
    GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")
except:
    raise RuntimeError("GEMINI_API_KEY not found in Kaggle Secrets")

genai.configure(api_key=GEMINI_API_KEY)


In [4]:
class Config:
    # Gemini model (fast + cheap for classification)
    MODEL_NAME = "gemini-2.5-flash"

    POS_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair.txt"
    NEG_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt"
    OUTPUT_FILE = "/kaggle/working/gemini_results.csv"
    OUTPUT_FILE2 = "/kaggle/working/gemini_fewshot_results.csv"

    SAMPLE_SIZE = 500
    RANDOM_SEED = 42

    # Gemini-specific
    TEMPERATURE = 0.0          # deterministic
    MAX_OUTPUT_TOKENS = 2048      # label only
    REQUEST_SLEEP = 0.2       # avoid rate limit

In [5]:
# =============================================================================
#  DATA LOADING & PREPARATION
# =============================================================================
FILENAME_PATTERN = re.compile(r"SupremeCourt_(\d{4})_(\d+)\.txt")

def parse_lrec_line(line: str):
    parts = line.strip().split("\t")
    fname_indices = [i for i, p in enumerate(parts) if p.endswith(".txt")]
    if len(fname_indices) != 2: return None
    try:
        sentpair_id = int(parts[0])
        sent1 = " ".join(parts[fname_indices[0] + 2 : fname_indices[1]]).strip('"')
        sent2 = " ".join(parts[fname_indices[1] + 2 : len(parts) - 2]).strip('"')
        label = parts[-1]
    except (ValueError, IndexError): return None
    return {"id": sentpair_id, "sent1": sent1, "sent2": sent2, "label": label}

def load_lrec_file(filepath: str) -> pd.DataFrame:
    rows = []
    print(f"Reading data from {filepath}...")
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if parsed_row := parse_lrec_line(line):
                rows.append(parsed_row)
    return pd.DataFrame(rows)

def load_and_split_data(config: Config):
    """
    Returns:
      df_test: test dataframe
      df_shots: pool from which few-shot exemplars are drawn (disjoint from df_test)
    """
    print("Loading and splitting data...")
    df_pos = load_lrec_file(config.POS_DATA_FILE)
    df_neg = load_lrec_file(config.NEG_DATA_FILE)
    if df_pos.empty or df_neg.empty:
        print("ERROR: Data files failed to load.")
        return None, None

    # Few-shot pool (sample a chunk to keep memory & speed reasonable)
    df_pos_pool = df_pos.sample(n=min(10000, len(df_pos)), random_state=config.RANDOM_SEED)
    df_neg_pool = df_neg.sample(n=min(10000, len(df_neg)), random_state=config.RANDOM_SEED)
    df_shots = pd.concat([df_pos_pool, df_neg_pool]).reset_index(drop=True)

    # Test set excludes the shot pool to avoid leakage
    remaining_pos = df_pos.drop(df_pos_pool.index)
    remaining_neg = df_neg.drop(df_neg_pool.index)
    df_test = pd.concat([remaining_pos, remaining_neg]).reset_index(drop=True)
    df_test = df_test.dropna(subset=['id', 'sent1', 'sent2', 'label'])
    print(f"Created test set with {len(df_test)} samples.")

    if config.SAMPLE_SIZE > 0:
        print(f"Using a sample of {config.SAMPLE_SIZE} examples from the test set for evaluation.")
        df_test = df_test.sample(n=min(config.SAMPLE_SIZE, len(df_test)), random_state=config.RANDOM_SEED).reset_index(drop=True)

    return df_test, df_shots


In [6]:
def gemini_generate(model, prompt, config):
    try:
        response = model.generate_content(
            prompt,
            generation_config={
                "temperature": config.TEMPERATURE,
                "max_output_tokens": config.MAX_OUTPUT_TOKENS,
            }
        )

        # Robust extraction
        if hasattr(response, "candidates") and len(response.candidates) > 0:
            parts = response.candidates[0].content.parts
            if parts and hasattr(parts[0], "text"):
                return parts[0].text

        # Fallback
        return ""

    except Exception as e:
        return f"__ERROR__::{str(e)}"

In [7]:
def make_zero_shot_prompt(sent1: str, sent2: str) -> str:
    return f"""
Analyze the relationship between the following two sentences.
Classify the relationship as exactly one of:
SUPPORT, ATTACK, or NO_REL.

Sentence 1: "{sent1}"
Sentence 2: "{sent2}"

Respond with ONLY the label.
"""

In [8]:
def extract_text_and_tokens(response):
    """
    Returns:
      text: str or None
      usage: dict with token counts
      finish_reason: str
    """
    text = None
    finish_reason = None

    if response.candidates:
        cand = response.candidates[0]
        finish_reason = cand.finish_reason

        if cand.content and cand.content.parts:
            part = cand.content.parts[0]
            if hasattr(part, "text"):
                text = part.text

    usage = {}
    if hasattr(response, "usage_metadata"):
        usage = {
            "prompt_tokens": response.usage_metadata.prompt_token_count,
            "output_tokens": response.usage_metadata.candidates_token_count,
            "total_tokens": response.usage_metadata.total_token_count,
        }

    return text, usage, finish_reason

In [9]:
class GeminiZeroShotClassifier:
    def __init__(self, config: Config):
        self.config = config
        self.model = genai.GenerativeModel(config.MODEL_NAME)

    def parse_output(self, text: str) -> str:
        text = text.strip().upper()
        if "SUPPORT" in text:
            return "SUPPORT"
        if "ATTACK" in text:
            return "ATTACK"
        if "NO_REL" in text or "NO RELATION" in text:
            return "NO_REL"
        return "NO_REL"  # safe fallback

    def predict(self, df: pd.DataFrame):
        predictions = []
        total_token = []
        output_token = []
        prompt_token = []

        for _, row in tqdm(df.iterrows(), total=len(df)):
            prompt = make_zero_shot_prompt(row.sent1, row.sent2)

            try:
                response = self.model.generate_content(
                    prompt,
                    generation_config={
                        "temperature": self.config.TEMPERATURE,
                        "max_output_tokens": self.config.MAX_OUTPUT_TOKENS,
                    }
                )
                
                pred = self.parse_output(response.text)
                text1, usage1, finish_reason1 = extract_text_and_tokens(response)
                temp_total_token = usage1['total_tokens']
                temp_output_token = usage1['output_tokens']
                temp_prompt_token = usage1['prompt_tokens']
                
            except Exception as e:
                pred = "NO_REL"
                temp_total_token = 0
                temp_output_token = 0
                temp_prompt_token = 0
                print("The error in ZeroShot is ", e)

            predictions.append(pred)
            total_token.append(temp_total_token)
            output_token.append(temp_output_token)
            prompt_token.append(temp_prompt_token)
            time.sleep(self.config.REQUEST_SLEEP)

        return predictions, total_token, output_token, prompt_token

In [10]:
def build_few_shot_block(examples):
    lines = ["Here are labeled examples:\n"]
    for i, ex in enumerate(examples, 1):
        lines.append(
            f"Example {i}:\n"
            f"Sentence 1: \"{ex['sent1']}\"\n"
            f"Sentence 2: \"{ex['sent2']}\"\n"
            f"Label: {ex['label']}\n"
        )
    lines.append(
        "Labels are one of: SUPPORT, ATTACK, NO_REL.\n"
        "Only output the label for the next pair.\n"
    )
    return "\n".join(lines)

In [11]:
def make_few_shot_prompt(sent1, sent2, examples_block):
    return f"""
You are a classifier for argumentative relations between two sentences.

{examples_block}

Now classify the following pair.
Respond with ONLY the label.

Sentence 1: "{sent1}"
Sentence 2: "{sent2}"
"""

In [12]:
class GeminiFewShotClassifier:
    def __init__(self, config: Config, exemplars: list):
        self.config = config
        self.model = genai.GenerativeModel(config.MODEL_NAME)
        self.examples_block = build_few_shot_block(exemplars)

    def parse_output(self, text: str) -> str:
        text = text.strip().upper()
        if "SUPPORT" in text:
            return "SUPPORT"
        if "ATTACK" in text:
            return "ATTACK"
        if "NO_REL" in text or "NO RELATION" in text:
            return "NO_REL"
        return "NO_REL"

    def predict(self, df: pd.DataFrame):
        predictions = []
        total_token = []
        output_token = []
        prompt_token = []

        for _, row in tqdm(df.iterrows(), total=len(df)):
            prompt = make_few_shot_prompt(
                row.sent1,
                row.sent2,
                self.examples_block
            )

            try:
                response = self.model.generate_content(
                    prompt,
                    generation_config={
                        "temperature": self.config.TEMPERATURE,
                        "max_output_tokens": self.config.MAX_OUTPUT_TOKENS,
                    }
                )
                
                pred = self.parse_output(response.text)
                text1, usage1, finish_reason1 = extract_text_and_tokens(response)
                temp_total_token = usage1['total_tokens']
                temp_output_token = usage1['output_tokens']
                temp_prompt_token = usage1['prompt_tokens']
            except Exception as e:
                print("The error in FewShot is ", e)
                pred = "NO_REL"
                temp_total_token = 0
                temp_output_token = 0
                temp_prompt_token = 0

            predictions.append(pred)
            total_token.append(temp_total_token)
            output_token.append(temp_output_token)
            prompt_token.append(temp_prompt_token)
            time.sleep(self.config.REQUEST_SLEEP)

        return predictions, total_token, output_token, prompt_token

In [13]:
if __name__ == "__main__":
    config = Config()
    df_test,df_shots = load_and_split_data(config)

    classifier = GeminiZeroShotClassifier(config)
    predictions, total_token, output_token, prompt_token = classifier.predict(df_test)

    true_labels = (
        df_test["label"]
        .str.upper()
        .str.replace(" ", "_")
        .replace({"NO_RELATION": "NO_REL"})
    )

    print("Accuracy:", accuracy_score(true_labels, predictions))
    print(classification_report(true_labels, predictions, zero_division=0))

    out = pd.DataFrame({
        "pair_id": df_test["id"],
        "sentence_1": df_test["sent1"],
        "sentence_2": df_test["sent2"],
        "original_label": df_test["label"],
        "gemini_label": predictions,
        "prompt_token": prompt_token,
        "output_token": output_token,
        "total_token": total_token
    })
    out.to_csv(config.OUTPUT_FILE, index=False)

Loading and splitting data...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair.txt...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt...
Created test set with 20506 samples.
Using a sample of 500 examples from the test set for evaluation.


100%|██████████| 500/500 [37:07<00:00,  4.46s/it]

Accuracy: 0.512
              precision    recall  f1-score   support

      ATTACK       0.48      0.41      0.44       130
      NO_REL       0.76      0.46      0.57       247
     SUPPORT       0.37      0.72      0.49       123

    accuracy                           0.51       500
   macro avg       0.54      0.53      0.50       500
weighted avg       0.59      0.51      0.52       500



In [14]:
time.sleep(60)

In [15]:
few_shot_examples = [
    {
        "sent1": "The tribunal misdirected itself in law.",
        "sent2": "The appeal is therefore allowed and the case remanded.",
        "label": "SUPPORT"
    },
    {
        "sent1": "The conviction relied on unlawful assembly.",
        "sent2": "The accused was acquitted under Section 148 IPC.",
        "label": "ATTACK"
    },
    {
        "sent1": "The defendant was guilty of perjury.",
        "sent2": "The Companies Act amendment comes into force next month.",
        "label": "NO_REL"
    }
]

config = Config()
df_test,df_shots = load_and_split_data(config)   # reuse your loader

clf = GeminiFewShotClassifier(config, few_shot_examples)
preds, total_token, output_token, prompt_token = clf.predict(df_test)

true_labels = (
    df_test["label"]
    .str.upper()
    .str.replace(" ", "_")
    .replace({"NO_RELATION": "NO_REL"})
)

print("Accuracy:", accuracy_score(true_labels, preds))
print(classification_report(true_labels, preds, zero_division=0))

out = pd.DataFrame({
        "pair_id": df_test["id"],
        "sentence_1": df_test["sent1"],
        "sentence_2": df_test["sent2"],
        "original_label": df_test["label"],
        "gemini_label": preds,
        "prompt_token": prompt_token,
        "output_token": output_token,
        "total_token": total_token
    })
out.to_csv(config.OUTPUT_FILE2, index=False)


Loading and splitting data...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair.txt...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt...
Created test set with 20506 samples.
Using a sample of 500 examples from the test set for evaluation.


100%|██████████| 500/500 [32:09<00:00,  3.86s/it]

Accuracy: 0.536
              precision    recall  f1-score   support

      ATTACK       0.48      0.47      0.47       130
      NO_REL       0.83      0.47      0.60       247
     SUPPORT       0.39      0.75      0.52       123

    accuracy                           0.54       500
   macro avg       0.57      0.56      0.53       500
weighted avg       0.63      0.54      0.54       500

